In [1]:
import os
import sys
from ultralytics import YOLO 
import yaml
import torch


# Add project root to system path (for relative imports to work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src import config

In [2]:

data_cfg = {
    'path': config.YOLO_DATA_DIR,
    'train': "images/train",
    'val': "images/val",
    'nc': 1,         # number of classes
    'names': ['motor']  # class names
}

with open(f'{config.SRC}/data.yaml', 'w') as f:
    yaml.dump(data_cfg, f)

print('data.yaml written successfully')

data.yaml written successfully


In [ ]:

model = YOLO('yolov8n.pt')
results = model.train(
    data=f'{config.SRC}/data.yaml',
    project=config.YOLO_RESULT,
    name='motor_detection_test',
    epochs=100,                # train longer
    batch=16,                  # smaller batch for high res
    imgsz=640,         # random between 640→1280
    multi_scale=True,          # enable multi-scale
    rect=False,
    device='cuda',             # if you have a GPU
    optimizer='Adam',
    lr0=1e-3,                  # lower initial LR
    lrf=0.1,                   # final LR = lr0 * lrf
    augment=True,              # mosaic + mixup
    cache=True,               # speeds up loading tiles
    patience=20,               # early-stop after 20 epochs no improvement
    save_period=1,             # save every epoch
    verbose=True,
    hsv_h=0.0, hsv_s=0.1, hsv_v=0.3,
  degrees=45, translate=0.2, scale=0.2, shear=5 ,
  fliplr=0.5, flipud=0.5, perspective=0.0001,
  mosaic=0.5, mixup=0.1, cutmix=0.05
)
print(results)


In [ ]:
model = YOLO('yolov8x.pt')
results = model.train(
    data=f'{config.SRC}/data.yaml',
    project=config.YOLO_RESULT,
    name='motor_detection_optimized',
    epochs=50,
    batch=16,
    imgsz=640,
    multi_scale=False,
    rect=False,
    device = 'cuda',

    # --- Optimizer & LR ---
    optimizer='SGD',
    lr0=0.005,               # lower initial LR for longer learning
    lrf=0.05,                # decay to 5% of lr0 at end
    momentum=0.937,
    weight_decay=5e-4,
    warmup_epochs=5,         # longer warmup before decay

    # --- Early stopping & checkpointing ---
    patience=8,              # stop after 8 epochs without improvement
    save_period=1,
    verbose=True,

    # --- Loss weights (rebalanced) ---
    box=0.10,                # up-weight box regression
    cls=0.20,                # down-weight classification
    dfl=1.50,

    # --- Augmentations ---
    augment=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.5,

    degrees=15,
    translate=0.1,
    scale=0.1,
    shear=2,
    fliplr=0.5,
    flipud=0.0,
    perspective=0.0,

    mosaic=0.5,
    mixup=0.1,
    cutmix=0.0,
)
print(results)


Ultralytics 8.3.162 🚀 Python-3.9.18 torch-2.7.1+cu126 CUDA:0 (NVIDIA A100-SXM4-40GB, 40446MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=0.1, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.2, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/h4/kein254g/BYU_Locating_Bacterial_Flagellar_Motors_2025/src/data.yaml, degrees=15, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.5, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.005, lrf=0.05, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=0.5, multi_scale=False, name=motor_detection_optimized13, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overla

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 39.50 GiB of which 1.81 MiB is free. Process 3758750 has 34.22 GiB memory in use. Including non-PyTorch memory, this process has 5.26 GiB memory in use. Of the allocated memory 4.70 GiB is allocated by PyTorch, and 41.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
model = YOLO('yolov8s.pt')
results = model.train(
    data=f'{config.SRC}/data.yaml',
    project=config.YOLO_RESULT,
    name='medical_detection_optimized',
    epochs=50,
    batch=8,                 # smaller batch for higher-res medical scans
    imgsz=1024,              # larger input size to capture fine structures
    multi_scale=True,        # allow varying image scales each epoch
    rect=True,               # respect original aspect ratios

    # --- Optimizer & LR for small datasets ---
    optimizer='Adam',        # adaptive updates often help on limited data
    lr0=1e-3,                # start lower for stability
    lrf=0.1,                 # final LR = 10% of lr0
    weight_decay=1e-5,       # light regularization

    # --- Early stopping & checkpoints ---
    patience=15,             # give more epochs to improve on scarce positives
    save_period=1,
    verbose=True,

    # --- Rebalanced loss weights for small, subtle objects ---
    box=0.20,                # more emphasis on precise box placement
    cls=0.20,                # balanced class loss so detection isn’t overwhelmed
    dfl=2.50,                # sharpen edge localization for tiny structures

    # --- Augmentations tuned for medical imagery ---
    augment=True,
    hsv_h=0.0,               # no hue shift (often grayscale or consistent staining)
    hsv_s=0.0,
    hsv_v=0.1,               # slight brightness/contrast variation
    degrees=45,              # full rotations to handle arbitrary orientations
    translate=0.2,           # small shifts to simulate framing variation
    scale=0.2,               # zoom in/out for variable magnification
    shear=5,                 # minor shearing for acquisition distortions
    fliplr=0.5,              # horizontal flips
    flipud=0.5,              # vertical flips (if anatomical orientation is arbitrary)
    perspective=0.0001,      # minimal perspective warp

    mosaic=0.5,              # moderate mosaic to combine multiple fields of view
    mixup=0.0,               # disable mixup (can create unrealistic overlays)
    cutmix=0.0,              # disable cutmix (avoid unnatural tissue cuts)
)
print(results)


In [ ]:
# Load a model
last_weight   = config.YOLO_TRAIN_RESULT + "/motor_detection_optimized/weights/last.pt"

model = YOLO(last_weight)  # load a partially trained model

# Resume training
results = model.train(resume=True, epochs=12)


In [ ]:
# 5.1 Load model and run inference

# — adjust these paths to your layout —
weights_path   = config.YOLO_TRAIN_RESULT + "/motor_detection_optimized/weights/best.pt"
val_image_dir  = config.VAL_IMAGE_DIR # your folder of val images
output_dir     = config.OUTPUT_DIR    # where to save annotated images

# Load the best weights
model = YOLO(weights_path)

# Run prediction on the entire folder, save annotated images & TXT
results = model.predict(
    project=config.YOLO_RESULT_PREDICT,
    source=val_image_dir,
    imgsz=640,
    conf=0.10,
    iou=0.03,
    max_det=100,
    save=True,
    save_dir=output_dir,
    save_txt=True
)


print(f"✅ Predictions saved to {output_dir}")


In [ ]:
# 5.1 Load model and run inference

# — adjust these paths to your layout —
weights_path   = config.YOLO_TRAIN_RESULT + "/motor_detection_optimized/weights/best.pt"
val_image_dir  = config.VAL_IMAGE_DIR # your folder of val images
output_dir     = config.OUTPUT_DIR    # where to save annotated images

# Load the best weights
model = YOLO(weights_path)

# Run prediction on the entire folder, save annotated images & TXT
results = model.val(
    project=config.YOLO_RESULT_PREDICT,
    source=val_image_dir,
    imgsz=640,
    conf=0.22,
    iou=0.03,
    max_det=100,
    save=True,
    save_dir=output_dir,
    save_txt=True
)


print(f"✅ Predictions saved to {output_dir}")
